# 🧹 DATA PREPROCESSING PIPELINE (5 BƯỚC CHUẨN ENGINE)
**Dự án:** Store-Level Sales Forecasting Engine  
**Phụ trách:** Tùng (Data Analyst)  
**Mục tiêu:** Tiền xử lý, gộp dữ liệu, tạo đặc trưng Time-series chống Data Leakage và phân tách 3 tập dữ liệu (`train_sub`, `val_set`, `test_final`).

# KHAI BÁO THƯ VIỆN

In [4]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import pandas as pd
from sklearn.preprocessing import LabelEncoder

print("✅ Đã khai báo thư viện thành công!")

✅ Đã khai báo thư viện thành công!


### BƯỚC 1: Đọc và nối tập Train + Test thành bảng tổng (`df_all`)
* **Mục đích:** Nối tập Train và Test theo chiều dọc để giữ chuỗi thời gian liên tục, phục vụ tạo biến Lag giáp ranh không bị khuyết (`NaN`).

In [7]:
import pandas as pd
from pathlib import Path

# ==============================================================================
# BƯỚC 1: ĐỌC VÀ NỐI TRAIN + TEST THÀNH BẢNG TỔNG (DF_ALL)
# =================================================----------------=============

# 1.1 Hàm dò đường dẫn thông minh: Tự tìm nơi thực sự chứa file 'train.csv'
def get_data_dir():
    current_dir = Path.cwd()
    
    # Danh sách các vị trí thư mục có thể chứa data
    search_paths = [
        current_dir / 'data' / 'raw',
        current_dir / 'data',
        current_dir.parent / 'data' / 'raw',
        current_dir.parent / 'data'
    ]
    
    # Dò qua các thư mục cha cấp cao hơn nếu notebook nằm sâu trong folder con
    for p in current_dir.parents:
        search_paths.append(p / 'data' / 'raw')
        search_paths.append(p / 'data')
        
    # Kiểm tra xem vị trí nào THỰC SỰ chứa file train.csv
    for path in search_paths:
        if (path / 'train.csv').exists():
            return path
            
    raise FileNotFoundError(
        "❌ Không tìm thấy file 'train.csv'! Hãy kiểm tra xem bạn đã chép các file .csv "
        "vào thư mục 'data/' hoặc 'data/raw/' trong dự án chưa."
    )

# Tự động xác định đường dẫn chuẩn xác
DATA_DIR = get_data_dir()
print(f"📁 Đường dẫn dữ liệu đã được tìm thấy chính xác tại: {DATA_DIR}")

# 1.2 Đọc các file dữ liệu thô
train = pd.read_csv(DATA_DIR / 'train.csv')  # Có cột Weekly_Sales
test = pd.read_csv(DATA_DIR / 'test.csv')    # Cột Weekly_Sales bị trống (NaN)

# 1.3 Nối 2 tập lại theo hàng dọc
df_all = pd.concat([train, test], axis=0, ignore_index=True)
df_all['Date'] = pd.to_datetime(df_all['Date'])

print(f"✅ [BƯỚC 1] Đã nối Train ({len(train)} dòng) và Test ({len(test)} dòng) -> Bảng tổng df_all: {len(df_all)} dòng.")

📁 Đường dẫn dữ liệu đã được tìm thấy chính xác tại: d:\AIO Tài liệu\OFFICIAL COURSE\Module 3\00. Project\Sales Forecasting and Demand Prediction\AIO_Conquer_Module3_PorscheClub\data
✅ [BƯỚC 1] Đã nối Train (421570 dòng) và Test (115064 dòng) -> Bảng tổng df_all: 536634 dòng.


### BƯỚC 2: GroupBy nén doanh số về cấp Cửa hàng (Store Level)
* **Mục đích:** Tiêu biến hoàn toàn nhiễu từ các Dept nhỏ và tình trạng đứt đoạn dòng thời gian bằng cách cộng dồn doanh số cấp Store.

In [ ]:
# --------------------------------------------------------------------------
# BƯỚC 2: GROUPBY NÉN DOANH SỐ VỀ CẤP STORE (CÒN ~8,000 DÒNG)
# --------------------------------------------------------------------------
df_store = df_all.groupby(['Store', 'Date', 'IsHoliday'], as_index=False)['Weekly_Sales'].sum()

print(f"✅ [BƯỚC 2] Đã nén doanh số về cấp Store. Tổng số dòng: {len(df_store)} dòng.")
print(f"   Khoảng thời gian: Từ {df_store['Date'].min().strftime('%Y-%m-%d')} đến {df_store['Date'].max().strftime('%Y-%m-%d')}")
df_store.head()

✅ [BƯỚC 2] Đã nén doanh số về cấp Store. Tổng số dòng: 8190 dòng.
   Khoảng thời gian: Từ 2010-02-05 đến 2013-07-26


,Store,Date,IsHoliday,Weekly_Sales
0,1,2010-02-05,False,1643690.90
1,1,2010-02-12,True,1641957.44
2,1,2010-02-19,False,1611968.17
3,1,2010-02-26,False,1409727.59
4,1,2010-03-05,False,1554806.68


### BƯỚC 3: Merge các bảng ngoại cảnh (`features.csv` & `stores.csv`)
* **Mục đích:** Bổ sung các yếu tố Vĩ mô (`CPI`, `Fuel_Price`, `Unemployment`) và Nội tại Cửa hàng (`Size`, `Type`) trên bảng gọn ~6,000 dòng để tối ưu bộ nhớ RAM.

In [9]:
# ==============================================================================
# BƯỚC 3: MERGE CÁC BẢNG NGOẠI CẢNH (FEATURES.CSV & STORES.CSV)
# ==============================================================================
features = pd.read_csv(DATA_DIR / 'features.csv')
stores = pd.read_csv(DATA_DIR / 'stores.csv')
features['Date'] = pd.to_datetime(features['Date'])

# Merge 1-1 trên cấp Store & Date
df_merged = df_store.merge(stores, on='Store', how='left')
df_merged = df_merged.merge(features, on=['Store', 'Date', 'IsHoliday'], how='left')

# Sắp xếp đúng thứ tự thời gian theo từng Store (Bắt buộc để tính Lag chính xác)
df_merged = df_merged.sort_values(by=['Store', 'Date']).reset_index(drop=True)

print(f"✅ [BƯỚC 3] Đã merge xong bảng ngoại cảnh. Kích thước DataFrame: {df_merged.shape}")

✅ [BƯỚC 3] Đã merge xong bảng ngoại cảnh. Kích thước DataFrame: (8190, 15)


### BƯỚC 4: Tạo các biến Time-Series (`Lag` & `Rolling`) chống Data Leakage
* **Mục đích:** Tạo bộ đặc trưng quá khứ bằng `.shift(1)` để đảm bảo tuyệt đối không lấy dữ liệu tương lai hoặc doanh số của chính tuần đó.

In [10]:
# --------------------------------------------------------------------------
# BƯỚC 4: TẠO CÁC BIẾN TIME-SERIES (LAG & ROLLING) TRÊN DF_MERGED
# --------------------------------------------------------------------------
# 4.1 Biến Lag Lịch sử (Dùng .shift(1) chống Data Leakage)
df_merged['Lag_1'] = df_merged.groupby('Store')['Weekly_Sales'].shift(1)
df_merged['Lag_52'] = df_merged.groupby('Store')['Weekly_Sales'].shift(52)

# 4.2 Biến Rolling Mean 4 tuần
df_merged['Rolling_Mean_4w'] = df_merged.groupby('Store')['Weekly_Sales'].transform(
    lambda x: x.shift(1).rolling(4).mean()
)

# 4.3 Trích xuất thêm đặc trưng Thời gian & Mã hóa
df_merged['Year'] = df_merged['Date'].dt.year
df_merged['Month'] = df_merged['Date'].dt.month
df_merged['WeekOfYear'] = df_merged['Date'].dt.isocalendar().week.astype(int)
df_merged['IsHoliday'] = df_merged['IsHoliday'].astype(int)

le = LabelEncoder()
df_merged['Type_encoded'] = le.fit_transform(df_merged['Type'])

# Forward fill cho các khuyết thiếu nhỏ ở biến Vĩ mô (nếu có)
macro_cols = ['CPI', 'Unemployment', 'Fuel_Price', 'Temperature']
df_merged[macro_cols] = df_merged.groupby('Store')[macro_cols].ffill()

print("✅ [BƯỚC 4] Đã tạo xong các đặc trưng Lag_1, Lag_52, Rolling_Mean_4w và biến Thời gian.")

✅ [BƯỚC 4] Đã tạo xong các đặc trưng Lag_1, Lag_52, Rolling_Mean_4w và biến Thời gian.


### BƯỚC 5: Lọc bỏ NaN lịch sử & Phân tách 3 tập dữ liệu (`train_sub`, `val_set`, `test_final`)
* **Mục đích:** 1. Loại bỏ 52 tuần đầu dính `NaN` khởi tạo ở `Lag_52`.
  2. Chia `train_final` thành `train_sub` (huấn luyện) và `val_set` (đánh giá RMSE & làm báo cáo).
  3. Giữ nguyên `test_final` đã sạch 100% đặc trưng để dự báo tương lai thực tế.

In [11]:
# --------------------------------------------------------------------------
# BƯỚC 5: CẮT THÀNH TRAIN_SUB, VAL_SET (ĐỂ BÁO CÁO) VÀ TEST_FINAL (ĐỂ DỰ BÁO)
# --------------------------------------------------------------------------
# 5.1. Xóa các dòng NaN ở 52 tuần đầu tiên do Lag_52 chưa có dữ liệu 2009
df_clean = df_merged.dropna(subset=['Lag_52']).reset_index(drop=True)

# 5.2. Lấy toàn bộ tập TRAIN ban đầu (những dòng có Weekly_Sales không phải NaN)
train_final = df_clean[df_clean['Weekly_Sales'].notna()].copy()

# 5.3. CẮT TRAIN_FINAL THÀNH TRAIN_SUB VÀ VAL_SET (Tập kiểm thử nội bộ)
VAL_SPLIT_DATE = '2012-05-01'

train_sub = train_final[train_final['Date'] < VAL_SPLIT_DATE].copy()
val_set   = train_final[train_final['Date'] >= VAL_SPLIT_DATE].copy()

# 5.4. Lấy tập TEST_FINAL ban đầu (những dòng có Weekly_Sales bị NaN - cần dự báo tương lai)
test_final = df_clean[df_clean['Weekly_Sales'].isna()].copy()

# --- IN THÔNG SỐ XÁC NHẬN BÁO CÁO (QC CHECK) ---
print("==========================================================================")
print("📊 BÁO CÁO KIỂM ĐỊNH TẬP DỮ LIỆU SAU KHI XỬ LÝ (QC CHECK)")
print("==========================================================================")
print(f"✅ [TRAIN_SUB]  : {len(train_sub)} dòng (Từ {train_sub['Date'].min().strftime('%Y-%m-%d')} đến {train_sub['Date'].max().strftime('%Y-%m-%d')})")
print(f"✅ [VAL_SET]    : {len(val_set)} dòng (Từ {val_set['Date'].min().strftime('%Y-%m-%d')} đến {val_set['Date'].max().strftime('%Y-%m-%d')}) -> DÙNG TÍNH RMSE & VẼ ĐỒ THỊ BÁO CÁO")
print(f"✅ [TEST_FINAL] : {len(test_final)} dòng -> ĐÃ SẠCH 100% FEATURE, SẴN SÀNG PREDICT TƯƠNG LAI")
print("==========================================================================")

📊 BÁO CÁO KIỂM ĐỊNH TẬP DỮ LIỆU SAU KHI XỬ LÝ (QC CHECK)
✅ [TRAIN_SUB]  : 2925 dòng (Từ 2011-02-04 đến 2012-04-27)
✅ [VAL_SET]    : 2925 dòng (Từ 2012-05-04 đến 2013-07-26) -> DÙNG TÍNH RMSE & VẼ ĐỒ THỊ BÁO CÁO
✅ [TEST_FINAL] : 0 dòng -> ĐÃ SẠCH 100% FEATURE, SẴN SÀNG PREDICT TƯƠNG LAI


In [14]:
# --------------------------------------------------------------------------
# BƯỚC 5 (ĐÓNG GÓI): XUẤT CÁC TỆP DỮ LIỆU SẠCH VÀO CÙNG THƯ MỤC DATA GỐC
# --------------------------------------------------------------------------

# Lưu trực tiếp vào DATA_DIR (Nơi chứa train.csv, features.csv, stores.csv)
train_sub.to_csv(DATA_DIR / 'train_final.csv', index=False)
val_set.to_csv(DATA_DIR / 'val_set.csv', index=False)
test_final.to_csv(DATA_DIR / 'test_final.csv', index=False)

print("==========================================================================")
print("🎉 [COMPLETED] Đã xuất thành công 3 tệp dữ liệu sạch vào chung thư mục gốc:")
print(f"📁 Đường dẫn lưu file: {DATA_DIR.resolve()}")
print("==========================================================================")

🎉 [COMPLETED] Đã xuất thành công 3 tệp dữ liệu sạch vào chung thư mục gốc:
📁 Đường dẫn lưu file: D:\AIO Tài liệu\OFFICIAL COURSE\Module 3\00. Project\Sales Forecasting and Demand Prediction\AIO_Conquer_Module3_PorscheClub\data


In [16]:
df1 = pd.read_csv(DATA_DIR / 'val_set.csv')
df1.head(10)

,Store,Date,IsHoliday,Weekly_Sales,Type,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,...,MarkDown5,CPI,Unemployment,Lag_1,Lag_52,Rolling_Mean_4w,Year,Month,WeekOfYear,Type_encoded
0,1,2012-05-04,0,1684519.99,A,151315,75.55,3.749,21290.13,NaN,...,3261.04,221.671800,7.143,1468928.37,1629391.28,1.627804e+06,2012,5,18,0
1,1,2012-05-11,0,1611096.05,A,151315,73.77,3.688,8351.40,NaN,...,3127.88,221.725663,7.143,1684519.99,1604775.58,1.574014e+06,2012,5,19,0
2,1,2012-05-18,0,1595901.87,A,151315,70.33,3.630,6154.14,NaN,...,5508.18,221.742674,7.143,1611096.05,1428218.27,1.571531e+06,2012,5,20,0
3,1,2012-05-25,0,1555444.55,A,151315,77.22,3.561,4039.39,NaN,...,3631.13,221.744944,7.143,1595901.87,1466046.67,1.590112e+06,2012,5,21,0
4,1,2012-06-01,0,1624477.58,A,151315,77.95,3.501,6086.21,12.00,...,3690.85,221.747214,7.143,1555444.55,1635078.41,1.611741e+06,2012,6,22,0
5,1,2012-06-08,0,1697230.96,A,151315,78.30,3.452,8813.81,116.80,...,7161.91,221.749484,7.143,1624477.58,1588948.32,1.596730e+06,2012,6,23,0
6,1,2012-06-15,0,1630607.00,A,151315,79.35,3.393,5621.99,109.60,...,3083.26,221.762642,7.143,1697230.96,1532114.86,1.618264e+06,2012,6,24,0
7,1,2012-06-22,0,1527845.81,A,151315,78.39,3.346,8624.56,171.25,...,7063.68,221.803021,7.143,1630607.00,1438830.15,1.626940e+06,2012,6,25,0
8,1,2012-06-29,0,1540421.49,A,151315,84.88,3.286,3965.73,161.60,...,4212.97,221.843400,7.143,1527845.81,1488538.09,1.620040e+06,2012,6,26,0
9,1,2012-07-06,0,1769854.16,A,151315,81.57,3.227,12218.76,94.40,...,6149.04,221.883779,6.908,1540421.49,1534849.64,1.599026e+06,2012,7,27,0
